# Amparo -- M1 v2: fine-tuning con pares contrastivos LoRA del agente legal

Este notebook:
1. Carga `dataset_m1_v2.jsonl` (**2291 ejemplos**: 1536 de v1 sin contexto y 755 de v2 con contexto recuperado) y usa el split que ya viene resuelto en el archivo. La validacion de v1 es la misma de siempre, asi que las cifras se pueden comparar con la corrida de M2 del 2026-10-06.
2. Corre el modelo base sin modificar sobre esa validacion (**baseline**).
3. Entrena un adaptador **LoRA** sobre el resto del dataset.
4. Vuelve a correr el modelo ya afinado sobre la misma validacion.
5. Compara baseline vs. afinado -- esa comparacion es la evidencia real de que el fine-tuning mejoro algo, no solo una suposicion.

**Sobre esta version del dataset.** La version anterior (1320 ejemplos) daba consejo generico sin decirle a la persona a donde acudir: solo el 14% de las respuestas nombraba un mecanismo legal concreto, lo que contradecia el principio 3 de `PRODUCT.md`. El dataset se reescribio para que cada respuesta señale la ruta real (tutela, Inspeccion del Trabajo, Comisaria de Familia, centro de conciliacion, etc.) y la evidencia a reunir; hoy lo cumple el **99.7%**. Tambien se agregaron tres categorias que faltaban y concentraban los casos de urgencia: servicios publicos domiciliarios, violencia intrafamiliar y penal basico/victimas.

**Que cambio con v2.** El dataset original no ensenaba numeros de articulo a proposito: ese conocimiento lo aporta el RAG de M3. Pero en la corrida del 2026-10-07 el modelo cito en **0 de 9** casos aun teniendo el articulo correcto delante (`results/m3_s08_2026-10-07`), porque nunca vio un ejemplo con contexto: 0 de 1536. Los 755 ejemplos de v2 ensenan la conducta que faltaba -- usar el CONTEXTO y decir de que norma sale -- sin tocar v1, que sigue ensenando a responder sin inventar cuando no hay contexto. Los dos modos traen system prompts distintos, asi que el modelo aprende la conducta condicional.

Antes de correr: `Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU` (una T4 gratis alcanza usando 4-bit/QLoRA).

**Nota sobre las librerias:** `transformers`, `peft` y `trl` cambian su API con cierta frecuencia. Este notebook esta escrito contra las versiones estables mas recientes conocidas al momento de escribirlo; si algun cambio de firma da error, revisa el changelog de la libreria correspondiente -- la logica de cada celda (que hace y por que) sigue siendo valida aunque cambie algun nombre de parametro.
---

**Que cambia frente a v1.** El adaptador v1 (`8ce3cc2bc9306974`) se abstuvo en
0 de 35 casos B2. El diagnostico esta en el commit `deee955`: de las 601
preguntas base con contexto en entrenamiento, solo UNA aparecia en mas de un
modo, asi que el modo era predecible desde la pregunta y el modelo podia
memorizarlo en vez de leer el contexto.

Esta corrida entrena con `data/dataset_m1_v3.jsonl`, que suma 418 variantes
contrastivas: la misma pregunta con un contexto que no responde. Las preguntas
vistas en varios modos pasan de 1 a 402 y la senal de abstencion de 8.3 % a
16.6 %.

**Los hiperparametros NO cambian** (r=16, alpha=32, 3 epocas, lr 2e-4, seed 42):
si cambian dos cosas a la vez, una mejora en B2 no se puede atribuir a los
pares contrastivos.

**El adaptador v1 no se toca.** Esta corrida escribe en
`amparo-lora-adapter-v2`, no en `amparo-lora-adapter`, que es la ruta que lee
`tools/rag/config.py`.

**La validacion es la misma**: 334 registros, los mismos ids que v1.


In [ ]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets wandb
!pip install -U "bitsandbytes>=0.46.1"

## Codigo y dataset desde el repo

In [ ]:
# El codigo y el dataset se toman del repo, no de Drive.
#
# Antes el dataset se leia de Drive y las metricas vivian copiadas dentro del
# notebook. Las dos cosas se desincronizaron con consecuencias reales: una
# version recortada del catalogo de rutas marcaba como "no nombra ruta"
# respuestas que si la nombraban, y nada fallaba a la vista. Clonando el repo
# hay una sola fuente de verdad para el dataset y para cada metrica.
RAMA = "m3.5"

import os

if not os.path.isdir("/content/Amparo"):
    !git clone -q -b {RAMA} https://github.com/TomasPosada0626/Amparo.git
else:
    !cd /content/Amparo && git fetch -q origin && git checkout -q {RAMA} && git pull -q
%cd /content/Amparo

!git log -1 --format="Codigo del commit %h (%ad)" --date=short


## Configuracion

Cambia `MODEL_ID` para comparar Qwen2.5 7B vs. Llama 3.1 8B (o el que haya ganado en el comparador local `tools/model_comparator`).

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"  # alternativa: "meta-llama/Llama-3.1-8B-Instruct"

VAL_FRACTION = 0.15   # % por categoria reservado para validacion (nunca se entrena con esto)
RANDOM_SEED = 42

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8   # batch efectivo = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS
# 1024 alcanzaba cuando el prompt era system + pregunta. Los ejemplos de v2
# traen ademas el CONTEXTO (hasta 5 fragmentos de norma), asi que a 1024 se
# truncarian justo por donde esta lo que deben ensenar a citar.
MAX_SEQ_LENGTH = 3072

# Era 300: a ese techo el baseline (que responde largo, con listas) quedaba
# cortado a mitad de frase en la mayoria de los casos, y la evaluacion lo
# castigaba por incompleto en vez de por incorrecto. 900 es el mismo valor
# que MAX_NEW_TOKENS_GENERATION de tools/evaluation/config.py, para que la
# cifra de M1 y la de M2 se midan bajo las mismas condiciones.
MAX_NEW_TOKENS_EVAL = 900
OUTPUT_DIR = "/content/amparo-lora"

## Monta tu Google Drive (solo para las salidas)

El dataset ya NO se sube a Drive: viene del repo clonado arriba, que es la
fuente de verdad. Antes se leia de Drive y eso permitia entrenar con una copia
vieja sin ningun error visible.

En Drive quedan solo las cosas que deben sobrevivir al reinicio de la maquina
de Colab: el adaptador entrenado y los resultados de la evaluacion.

El dataset no se edita a mano: se construye desde las fuentes versionadas en
`data/dataset_src/` con `python -m tools.dataset_build`, que solo lo escribe si
pasa las ocho puertas de calidad de `tools/dataset_quality.py`. Asi queda
trazable con que datos se entreno y cualquier cambio es revisable ejemplo por
ejemplo en un diff.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# El dataset sale del repo clonado (fuente de verdad). En Drive quedan solo las
# SALIDAS, que si deben sobrevivir al reinicio de la maquina de Colab.
# El combinado v1+v2+contrastivo (tools/dataset_m1_v3.py): 1536 sin contexto,
# 755 con contexto y 418 variantes contrastivas. Los splits vienen resueltos y
# NO se recalculan: la validacion tiene que seguir siendo la de v1.
DATASET_PATH = '/content/Amparo/data/dataset_m1_v3.jsonl'
DATASET_HASH_ESPERADO = '79bcc7e328a1f146'

DRIVE_DIR = '/content/drive/MyDrive/Colab Notebooks/Amparo'
SALIDA_DIR = f'{DRIVE_DIR}/evaluacion'   # resultados de la corrida, no en /content
# Con fecha: el adaptador sin version se sobrescribio el 6 de octubre y las
# corridas del 27 de septiembre quedaron sin forma de reproducirse. La carpeta
# sin fecha se sigue actualizando porque tools/rag/config.py apunta ahi, pero
# la fechada queda como respaldo.
from datetime import date
VERSION_ADAPTADOR = date.today().isoformat()
# v2 escribe en su propia carpeta. La ruta sin version
# (amparo-lora-adapter) es la que lee tools/rag/config.py y la que usan M2 y
# M3: si esta corrida escribiera ahi, sobrescribiria el adaptador v1 y no
# quedaria baseline contra el que comparar. Se cambia a mano, cuando se decida
# adoptar v2.
ADAPTER_DIR = f'{DRIVE_DIR}/amparo-lora-adapter-v2'
ADAPTER_DIR_VERSIONADO = f'{DRIVE_DIR}/amparo-lora-adapter-v2-{VERSION_ADAPTADOR}'

import os
os.makedirs(SALIDA_DIR, exist_ok=True)
print(f'Dataset (del repo): {DATASET_PATH}')
print(f'Resultados -> {SALIDA_DIR}')
print(f'Adaptador   -> {ADAPTER_DIR}')
assert 'v2' in ADAPTER_DIR, 'el adaptador v1 no se sobrescribe'


## Cargar y separar train / validacion (estratificado por categoria)

In [ ]:
import json
import random
from collections import Counter, defaultdict

random.seed(RANDOM_SEED)

records = [json.loads(line) for line in open(DATASET_PATH, encoding='utf-8')]

by_category = defaultdict(list)
for r in records:
    by_category[r['category']].append(r)

# El split viene resuelto en el archivo (tools/dataset_m1_v2.py) y NO se
# recalcula aca. v1 se partio con stratified_split -- verificado que da la misma
# validacion de 231 que hacia este notebook, asi que las cifras se siguen
# pudiendo comparar con M2 del 2026-10-06 -- y v2 con split_v2, que manda cada
# ejemplo al lado de su pregunta base: la misma pregunta sin contexto en train y
# con contexto en val haria que val midiera memoria.
train_records = [r for r in records if r['split'] == 'train']
val_records = [r for r in records if r['split'] == 'val']
random.shuffle(train_records)
random.shuffle(val_records)

print(f'Total: {len(records)} | Train: {len(train_records)} | Validacion: {len(val_records)}')
# Guarda contra el fallo silencioso de entrenar con una copia vieja del dataset
# que haya quedado en Drive: sin esto, el notebook corre igual y nadie se entera.
# El hash del archivo, antes que nada. Contar ejemplos no basta: un dataset
# regenerado con otra version del codigo puede tener los mismos 2709 registros y
# contextos distintos, y entonces esta corrida no seria comparable con la que se
# valido. Si el hash no coincide, no se entrena.
import hashlib

_hash = hashlib.sha1(open(DATASET_PATH, 'rb').read()).hexdigest()[:16]
assert _hash == DATASET_HASH_ESPERADO, (
    f'El dataset en disco tiene hash {_hash} y se esperaba '
    f'{DATASET_HASH_ESPERADO}. Regeneralo con: python -m tools.dataset_m1_v3'
)
print(f'  dataset {DATASET_PATH.split("/")[-1]} hash {_hash} -- el validado')

assert len(records) == 2709, (
    f'El dataset tiene {len(records)} ejemplos y se esperaban 2709 (1536 de v1, '
    f'755 de v2 y 418 contrastivas). Regeneralo con: python -m tools.dataset_m1_v3'
)
_origenes = Counter(r['origen'] for r in records)
assert _origenes == {'v1': 1536, 'v2': 755, 'contrastivo': 418}, (
    f'Mezcla inesperada de origenes: {dict(_origenes)}. Sin las contrastivas '
    f'esta corrida repite v1 y no prueba nada.'
)

# La validacion tiene que ser EXACTAMENTE la de v1 o la comparacion no vale.
assert len(val_records) == 334, (
    f'La validacion tiene {len(val_records)} y no 334: no se puede comparar con v1.'
)
assert all(r['origen'] != 'contrastivo' for r in val_records), (
    'Hay variantes contrastivas en validacion: se mediria sobre lo que entreno.'
)

_modos = Counter(r.get('modo') for r in records if r['origen'] != 'v1')
print(f'  v1 {_origenes["v1"]} sin contexto | con contexto {dict(_modos)}')

# Lo que esta corrida viene a cambiar, medido sobre el archivo que se va a usar.
_por_base = {}
for r in train_records:
    if r.get('modo'):
        _por_base.setdefault(r.get('base_id'), set()).add(r['modo'])
_multi = sum(1 for s in _por_base.values() if len(s) > 1)
_masa = lambda rs: sum(len(r['messages'][-1]['content']) for r in rs)
_b2 = _masa([r for r in train_records if r.get('modo') == 'B2'])
print(f'  preguntas vistas en mas de un modo: {_multi} (en v1 era 1)')
print(f'  senal de abstencion: {_b2 / _masa(train_records):.1%} (en v1 era 8.3%)')

for category in sorted(by_category):
    n_val = sum(1 for r in val_records if r['category'] == category)
    n_train = sum(1 for r in train_records if r['category'] == category)
    print(f'  {category:45s} train={n_train:3d}  val={n_val:3d}')

## Cargar el modelo base en 4-bit (QLoRA)

Se carga cuantizado en 4-bit para que quepa comodamente en una GPU gratuita de Colab (T4, ~16GB).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Que revision del modelo base se bajo.
#
# Ni v1 ni v2 fijan `revision`, asi que Hugging Face sirve lo que apunte main.
# Si Qwen publico un commit entre la corrida de v1 (2026-10-09) y esta, el
# modelo base no es el mismo y la comparacion queda contaminada. El manifiesto
# de v1 no lo registro, asi que no se puede saber cual uso.
#
# Aqui al menos queda anotado para las corridas que vengan. La comprobacion de
# verdad es empirica y esta mas abajo: con do_sample=False la decodificacion es
# determinista, asi que si el modelo base es el mismo el baseline tiene que
# salir identico al de v1.
try:
    from huggingface_hub import model_info

    REVISION_BASE = model_info(MODEL_ID).sha
except Exception as _e:
    REVISION_BASE = f'no se pudo resolver: {_e}'
print(f'Modelo base {MODEL_ID}')
print(f'  revision resuelta: {REVISION_BASE}')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)

## Metricas de dominio

Se importan del repo (`tools/`), no van copiadas aqui: una sola definicion por
propiedad, la misma que usan las puertas del dataset y el harness de M2.

- **Nombra ruta legal** (`tools/dataset_quality.py`): principio 3 de PRODUCT.md.
- **Cita normas inventadas** (`tools/evaluation/domain_metric.py`): la misma
  definicion que M2, para que las cifras sean comparables.
- **Inventa entidades** (`tools/evaluation/entity_metric.py`): el segundo modo
  de fabricacion, al que el regex de citas es ciego porque no lleva numero.

**Por que esta cifra puede no coincidir con la de M2.** Aqui se mide el modelo
que quedo en memoria despues de entrenar; M2 recarga el adaptador desde Drive
sobre el modelo base, normalmente en otra GPU. Con decodificacion greedy
cualquier diferencia numerica hace que el texto diverja desde el primer token
ambiguo. Las dos mediciones son legitimas pero no identicas, y la de referencia
es la de M2, por estar respaldada por un artefacto comiteado con su manifiesto.


In [ ]:
import json
import re

# Las metricas se IMPORTAN del repo: una sola definicion por propiedad.
#   - nombra_mecanismo: el mismo catalogo con el que se validaron las puertas
#     del dataset (tools/dataset_quality.py).
#   - cita_no_verificable: la misma definicion que usa el harness de M2
#     (tools/evaluation/domain_metric.py), para que las cifras sean comparables.
#   - entidades_inventadas: el segundo modo de fabricacion
#     (tools/evaluation/entity_metric.py).
#
# Antes estas definiciones iban copiadas dentro del notebook y la copia se
# desincronizo del repo, subestimando al modelo sin que nada fallara.
from tools.dataset_quality import menciona_mecanismo as nombra_mecanismo
from tools.evaluation.domain_metric import has_invented_citation as cita_no_verificable
from tools.evaluation.entity_metric import find_fabricated_entities

SYSTEM_PROMPT = records[0]['messages'][0]['content']

# --- Metricas de dominio ---------------------------------------------------
# Miden lo que el producto promete y son verificables sin juez LLM.
#
# 1) Nombrar la ruta legal concreta (principio 3 de PRODUCT.md). OJO al leerla:
#    esta saturada (el modelo base ya venia en 96%) y esta confundida con la
#    longitud, porque es coincidencia de palabras clave y el baseline escribe
#    ~5x mas texto, o sea ~5x mas superficie para acertar por azar. No sirve
#    para demostrar mejora en ninguna direccion.
# 2) Citar normas por numero. El dataset no las ensena a proposito -- ese
#    conocimiento lo aporta el RAG de M3 contra el corpus real -- asi que toda
#    cita que aparezca sale de la memoria del modelo.
# 3) Inventar entidades. Aparecio midiendo la corrida del 2026-10-05: el
#    adaptador dejo de inventar normas y seguia mandando a ventanillas que no
#    existen ("Superintendencia de Pensiones", "querella de omision"). Es el
#    mismo pecado y el regex de citas es ciego a el porque no llevan numero.
#
# Ya NO se calcula similitud lexica con difflib: daba 3.2 contra 6.0 sobre 100,
# no distingue una respuesta correcta de una incorrecta (solo fraseo) y la
# comparacion contra la referencia la hace M2 en serio, con BLEU/ROUGE/BERTScore
# sobre este mismo split.


def entidades_inventadas(texto: str) -> list:
    return find_fabricated_entities(texto)


# Campos que debe traer cada registro guardado. Se usa para rechazar
# checkpoints de corridas con otro esquema (p. ej. los que traian 'similarity').
# 'cortada' entra aqui a proposito: un checkpoint de una corrida que no lo
# registraba no se puede reusar, porque quedaria sin la evidencia de
# truncamiento que esta corrida viene a dejar.
CAMPOS_ESPERADOS = {'generated', 'nombra_mecanismo', 'cita_no_verificable',
                    'entidades_inventadas', 'n_tokens', 'cortada',
                    'motivo_fin', 'presupuesto_agotado'}


# Los ids que cuentan como fin de secuencia. La logica vive en
# tools/evaluation/fin_de_generacion.py para poder probarla: aqui dentro no se
# puede, y el detalle que importa es facil de equivocar -- en Qwen2.5-Instruct
# el token que cierra el turno es <|im_end|>, mientras tokenizer.eos_token_id
# apunta a <|endoftext|>, que el modelo instruido casi nunca emite. Mirando
# solo el eos del tokenizer, casi todas las respuestas saldrian cortadas.
from tools.evaluation.fin_de_generacion import ids_de_fin, motivo as _motivo

IDS_FIN = ids_de_fin(tokenizer, model)
assert IDS_FIN, 'sin token de fin no se puede saber si una respuesta se corto'
print(f'Tokens de fin de secuencia: {sorted(IDS_FIN)}')
print(f'  eos del tokenizer: {tokenizer.eos_token_id} '
      f'({tokenizer.convert_ids_to_tokens(tokenizer.eos_token_id)})')


@torch.no_grad()
def generate_response(model, query: str) -> tuple[str, int, str]:
    """(texto, tokens generados).

    Devuelve el conteo porque sin el no se puede saber si la respuesta se corto
    al agotar MAX_NEW_TOKENS_EVAL. En v1 solo devolvia el texto y, al cerrar el
    modulo, el truncamiento hubo que descartarlo acotandolo por longitud de
    palabras en vez de medirlo.
    """
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': query},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS_EVAL,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated = output[0][inputs['input_ids'].shape[1]:]
    texto = tokenizer.decode(generated, skip_special_tokens=True).strip()
    # Se le pasan SOLO los tokens nuevos: con el prompt concatenado, una
    # respuesta vacia leeria el ultimo token del prompt y pasaria por terminada.
    return texto, len(generated), _motivo(generated, IDS_FIN, MAX_NEW_TOKENS_EVAL)


def evaluate(model, val_set, label: str, checkpoint: str):
    # Persiste CADA ejemplo a Drive en el momento de generarlo, y si el archivo
    # ya tiene resultados los reusa y sigue desde ahi. Antes la escritura
    # ocurria solo al terminar el lote: con MAX_NEW_TOKENS_EVAL en 900 la
    # evaluacion del baseline toma horas y una desconexion de Colab borraba
    # todo el trabajo de la corrida.
    import os
    import time

    ruta = f'{SALIDA_DIR}/{checkpoint}'
    results = []
    if os.path.exists(ruta):
        with open(ruta, encoding='utf-8') as f:
            results = [json.loads(linea) for linea in f if linea.strip()]
        # Guarda de identidad: en Drive quedan checkpoints de corridas
        # anteriores (otro adaptador, otro dataset, u otro esquema de campos).
        # Reusarlos mezclaria dos corridas en una sola cifra sin que nada falle.
        if len(results) > len(val_set):
            raise SystemExit(
                f'El checkpoint evaluacion/{checkpoint} tiene {len(results)} registros y '
                f'la validacion solo {len(val_set)}: es de otra corrida. Borralo de Drive.')
        if [r['id'] for r in results] != [item['id'] for item in val_set[:len(results)]]:
            raise SystemExit(
                f'El checkpoint evaluacion/{checkpoint} no corresponde a esta validacion '
                f'(ids distintos a los esperados). Es de una corrida anterior: borralo de Drive.')
        if results and not CAMPOS_ESPERADOS.issubset(results[0]):
            faltan = CAMPOS_ESPERADOS - set(results[0])
            raise SystemExit(
                f'El checkpoint evaluacion/{checkpoint} es de un esquema anterior (le faltan '
                f'{sorted(faltan)}). Borralo de Drive para medir todo con las metricas vigentes.')
        if results:
            print(f'[{label}] reusando {len(results)} resultados ya guardados '
                  f'en evaluacion/{checkpoint}', flush=True)
    ya_hechos = len(results)
    t0 = time.perf_counter()
    for i, item in enumerate(val_set, start=1):
        if i <= ya_hechos:
            continue
        query = item['messages'][1]['content']
        actual, n_generados, motivo_fin = generate_response(model, query)
        inventadas = entidades_inventadas(actual)
        fila = {
            **item,
            'generated': actual,
            'nombra_mecanismo': nombra_mecanismo(actual),
            'cita_no_verificable': cita_no_verificable(actual),
            'entidades_inventadas': inventadas,
            'palabras': len(actual.split()),
            # Cuantos tokens salieron y si se agoto el presupuesto. En v1 no se
            # registro, y al cerrar el modulo hubo que descartar el truncamiento
            # acotandolo por longitud de palabras. tools/evaluation/generation.py
            # lo trae desde el principio; este notebook tiene su propio bucle y
            # le faltaba.
            'n_tokens': int(n_generados),
            # cortada: el modelo NO emitio el token de fin. Es la senal
            # fiable, y la que usa la metrica.
            'cortada': motivo_fin != 'termino',
            # Y el motivo aparte, porque las causas se arreglan distinto: una
            # salida vacia es un fallo de generacion y un presupuesto agotado es
            # un techo mal puesto, pero las dos cuentan como cortada.
            'motivo_fin': motivo_fin,
            'presupuesto_agotado': bool(n_generados >= MAX_NEW_TOKENS_EVAL),
        }
        results.append(fila)
        with open(ruta, 'a', encoding='utf-8') as f:
            f.write(json.dumps(fila, ensure_ascii=False) + '\n')
        hechos = i - ya_hechos
        transcurrido = time.perf_counter() - t0
        eta = transcurrido / hechos * (len(val_set) - i) / 60
        print(f"[{label}] {i}/{len(val_set)}  "
              f"ruta={'si' if fila['nombra_mecanismo'] else 'no '}  "
              f"{transcurrido / hechos:.1f}s/ej  ETA ~{eta:.1f} min", flush=True)
    n = len(results)
    print(f"\n[{label}] ruta legal: {sum(r['nombra_mecanismo'] for r in results) / n:.1%}"
          f" | citas numeradas: {sum(r['cita_no_verificable'] for r in results) / n:.1%}"
          f" | entidades inventadas: {sum(1 for r in results if r['entidades_inventadas']) / n:.1%}"
          f" | {sum(r['palabras'] for r in results) / n:.0f} palabras")
    return results


def guardar_resultados(nombre, datos):
    ruta = f'{SALIDA_DIR}/{nombre}'
    with open(ruta, 'w', encoding='utf-8') as f:
        for r in datos:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(f'Guardado en Drive: evaluacion/{nombre} ({len(datos)} registros)')


## Paso 1 -- Baseline (modelo base, sin fine-tuning)

Esto puede tardar varios minutos segun el tamano de la validacion.

In [ ]:
baseline_results = evaluate(model, val_records, label='baseline',
                            checkpoint='baseline_results.jsonl')
guardar_resultados('baseline_results.jsonl', baseline_results)

### Comprobacion: el modelo base es el mismo que en v1

Con `do_sample=False` la decodificacion es determinista, asi que el mismo modelo
base mas el mismo prompt dan la misma respuesta, caracter por caracter. Si el
baseline de esta corrida coincide con el de `results/m1_2026-10-09`, el modelo
y el tokenizer son los mismos y la comparacion v1 contra v2 es valida.

Si NO coincide, Hugging Face sirvio otra revision y las cifras de v1 no se
pueden poner al lado de las de v2 sin decirlo.


In [ ]:
# El baseline de v1, para comparar contra el que acaba de generarse.
import json as _json
from pathlib import Path as _P

_ruta_v1 = _P('/content/Amparo/results/m1_2026-10-09/baseline_results.jsonl')
if not _ruta_v1.exists():
    print(f'No esta {_ruta_v1}: no se puede comprobar el modelo base.')
else:
    _v1 = {r['id']: r['generated'] for r in
           (_json.loads(l) for l in _ruta_v1.read_text(encoding='utf-8').splitlines() if l.strip())}
    _ahora = {r['id']: r['generated'] for r in baseline_results}
    _comunes = sorted(set(_v1) & set(_ahora))
    _iguales = [i for i in _comunes if _v1[i] == _ahora[i]]
    print(f'Baseline comparado con v1 sobre {len(_comunes)} ids comunes:')
    print(f'  identicos caracter por caracter: {len(_iguales)}/{len(_comunes)} '
          f'({len(_iguales) / len(_comunes):.1%})')
    if len(_iguales) == len(_comunes):
        print()
        print('  El modelo base y el tokenizer son los mismos que en v1.')
        print('  La comparacion v1 contra v2 es valida.')
    else:
        _d = [i for i in _comunes if _v1[i] != _ahora[i]]
        print()
        print(f'  AVISO: {len(_d)} respuestas distintas. Primeros ids: {_d[:8]}')
        print('  Hugging Face pudo servir otra revision del modelo base, o cambio')
        print('  algo del tokenizer o de los parametros de generacion. Las cifras')
        print('  de v1 NO se pueden poner al lado de las de v2 sin decirlo.')
        print(f'  Revision de esta corrida: {REVISION_BASE}')


## Paso 2 -- Fine-tuning con LoRA

### Antes de entrenar: conecta Weights & Biases

M1 pide dejar registrada la curva de perdida del entrenamiento como evidencia visual de que el modelo esta aprendiendo. Crea una cuenta gratuita en wandb.ai si no tienes una, y ten a mano tu API key (la consigues en https://wandb.ai/authorize).

**Si corres este notebook desde VSCode** (extension de Colab) en vez del navegador: `wandb.login()` te pide la key en un recuadro de entrada; si se queda esperando sin mostrarlo, usa `wandb.login(key="tu_key")` en su lugar. Este notebook no depende de los Secrets de Colab (`userdata.get`), que es lo unico que no funciona fuera del navegador.

In [ ]:
import wandb

wandb.login()  # pega tu API key cuando te la pida (solo la primera vez)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from datasets import Dataset

# Dev para elegir la epoca, recortado de TRAIN con el mismo metodo estratificado.
# NO se usa la validacion de 231 para esto: si el checkpoint se elige mirandola,
# deja de ser held-out y las cifras del reporte quedan contaminadas.
DEV_FRACTION = 0.08

random.seed(RANDOM_SEED)
_por_categoria = defaultdict(list)
for r in train_records:
    _por_categoria[r['category']].append(r)

fit_records, dev_records = [], []
for _categoria, _items in sorted(_por_categoria.items()):
    _items = _items[:]
    random.shuffle(_items)
    _n_dev = max(1, round(len(_items) * DEV_FRACTION))
    dev_records.extend(_items[:_n_dev])
    fit_records.extend(_items[_n_dev:])
random.shuffle(fit_records)
random.shuffle(dev_records)


# Formato prompt/completion: TRL enmascara el prompt y calcula la perdida SOLO
# sobre la respuesta del asistente.
#
# Antes se entrenaba con el texto ya renderizado (dataset_text_field='text'),
# asi que la perdida incluia el system prompt -- 87 palabras IDENTICAS en los
# 1536 ejemplos, el 60% del presupuesto -- mas la pregunta. Solo el 32% del
# gradiente caia sobre lo unico que el modelo debe aprender a generar, y lo que
# el modelo ajustaba con fuerza era el estilo.
#
# No se usa assistant_only_loss=True: ese camino necesita que el chat template
# traiga marcadores {% generation %} para construir la mascara del turno del
# asistente, y el de Qwen2.5-Instruct no los tiene. El formato prompt/completion
# no depende de la plantilla.
def a_prompt_completion(record):
    mensajes = record['messages']
    return {'prompt': mensajes[:2], 'completion': mensajes[2:]}


# El combinado trae ademas las columnas de v2 (modo, base_id, contexto,
# fuentes...), utiles para auditar pero no para entrenar. Se arma una copia
# delgada: Dataset.from_list tendria que inferir el esquema de 'contexto', que
# es una lista de diccionarios, y las columnas que sobran rompen el enmascarado
# de TRL (lo atrapa el assert de abajo, pero mejor no llegar ahi).
_PARA_SFT = ('id', 'category', 'messages')


def _delgado(registros):
    return [{k: r[k] for k in _PARA_SFT} for r in registros]


_COLUMNAS_ORIGINALES = list(_PARA_SFT)
train_dataset = Dataset.from_list(_delgado(fit_records)).map(
    a_prompt_completion, remove_columns=_COLUMNAS_ORIGINALES)
dev_dataset = Dataset.from_list(_delgado(dev_records)).map(
    a_prompt_completion, remove_columns=_COLUMNAS_ORIGINALES)

print(f'Entrenamiento: {len(train_dataset)} | Dev (eleccion de epoca): {len(dev_dataset)}')
print(f'Validacion intacta (solo para el reporte final): {len(val_records)}')
print('Columnas que ve TRL:', train_dataset.column_names)
assert train_dataset.column_names == ['prompt', 'completion'], (
    'TRL necesita exactamente las columnas prompt y completion para enmascarar '
    'el prompt. Con otras columnas vuelve a entrenar sobre el texto completo.')


In [ ]:
wandb.init(
    project='amparo-legal-finetune',
    name=f"lora-{MODEL_ID.split('/')[-1]}",
    config={
        'model_id': MODEL_ID,
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'epochs': NUM_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'perdida': 'solo sobre la respuesta (prompt/completion)',
        'n_fit': len(train_dataset),
        'n_dev': len(dev_dataset),
    },
)

from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    # Evaluacion por epoca sobre el dev y se conserva el mejor checkpoint.
    # Antes no habia eval_dataset: solo se registraba la perdida de
    # entrenamiento, no habia forma de ver sobreajuste y las 3 epocas eran un
    # numero elegido, no medido -- se usaba el checkpoint de la ultima por
    # omision.
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=2,
    bf16=True,
    max_length=MAX_SEQ_LENGTH,
    # Explicito: antes la reproducibilidad del entrenamiento dependia del
    # default de transformers, no de una decision escrita.
    seed=RANDOM_SEED,
    report_to='wandb',
    run_name=f"lora-{MODEL_ID.split('/')[-1]}",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    processing_class=tokenizer,
)
print('Trainer listo. Verifica el enmascarado en la celda siguiente ANTES de entrenar.')


### Verificacion: la perdida cae solo sobre la respuesta

In [ ]:
# Verificacion del enmascarado, antes de gastar GPU.
#
# Esta celda existe porque el cambio a prompt/completion falla en silencio: si
# TRL no reconoce el formato, entrena sobre el texto completo igual que antes y
# el notebook corre sin un solo error. Lo unico que lo delata es la proporcion
# de tokens con etiqueta -100 (ignorados por la perdida).
import torch

_lote = next(iter(trainer.get_train_dataloader()))
_etiquetas = _lote['labels']
_ignorados = int((_etiquetas == -100).sum())
_aprendidos = int((_etiquetas != -100).sum())
_total = _ignorados + _aprendidos

print(f'Tokens del lote: {_total}')
print(f'  ignorados por la perdida (prompt): {_ignorados} ({100 * _ignorados / _total:.0f}%)')
print(f'  con perdida (respuesta):           {_aprendidos} ({100 * _aprendidos / _total:.0f}%)')

# Si el prompt NO se esta enmascarando, casi nada queda en -100 (solo el
# padding), y eso es exactamente el bug que esta celda busca.
assert _ignorados > _aprendidos, (
    f'El prompt no se esta enmascarando: solo {100 * _ignorados / _total:.0f}% de los '
    'tokens quedan fuera de la perdida. Con el system prompt de 87 palabras se '
    'espera que la mayoria del lote este ignorada. Revisa que train_dataset '
    'tenga solo las columnas prompt y completion.')
print('\nOK: la perdida se calcula solo sobre la respuesta del asistente.')


### Entrenar

In [ ]:
trainer.train()

print('Dashboard de W&B (guarda este link para tu informe de M1):', wandb.run.url)
wandb.finish()

# El historial de perdidas, a Drive y en texto.
#
# En v1 esto solo quedo en W&B. Al cerrar el modulo hizo falta la tabla de
# train_loss / eval_loss / epoca elegida y no se pudo recuperar: el objeto
# trainer vivia en una sesion de Colab que ya no existe, y no se puede imprimir
# desde otra. Aqui se escribe en el momento.
import json as _json

_hist = trainer.state.log_history
with open(f'{SALIDA_DIR}/historial_entrenamiento_v2.json', 'w', encoding='utf-8') as f:
    _json.dump({
        'log_history': _hist,
        'best_model_checkpoint': trainer.state.best_model_checkpoint,
        'best_metric': trainer.state.best_metric,
        'epoch': trainer.state.epoch,
        'global_step': trainer.state.global_step,
    }, f, ensure_ascii=False, indent=2)
print(f'Historial -> {SALIDA_DIR}/historial_entrenamiento_v2.json')
print()

print(f"{'epoca':>7}{'train_loss':>13}{'eval_loss':>12}")
print('-' * 32)
_por_epoca = {}
for _e in _hist:
    _ep = _e.get('epoch')
    if _ep is None:
        continue
    _d = _por_epoca.setdefault(round(_ep, 2), {})
    for _k in ('loss', 'eval_loss'):
        if _k in _e:
            _d[_k] = _e[_k]
for _ep in sorted(_por_epoca):
    _d = _por_epoca[_ep]
    _tr = f"{_d['loss']:.4f}" if 'loss' in _d else '-'
    _ev = f"{_d['eval_loss']:.4f}" if 'eval_loss' in _d else '-'
    print(f'{_ep:>7}{_tr:>13}{_ev:>12}')
print()
print(f'Mejor checkpoint por eval_loss: {trainer.state.best_model_checkpoint}')
print(f'  eval_loss = {trainer.state.best_metric}')
print()
print('Una perdida mas baja NO es la prueba de que v2 sea mejor: lo que decide')
print('es B2 sin perjudicar B1, B3, las citas ni los gold de M3.')


## Guardar el adaptador LoRA en Drive

Se guarda directo en tu Drive (no solo son unos MB, son los pesos LoRA) para que no se pierda si el entorno de Colab se desconecta a mitad del entrenamiento o despues.

In [ ]:
import shutil

model.save_pretrained(f'{OUTPUT_DIR}/adapter')
tokenizer.save_pretrained(f'{OUTPUT_DIR}/adapter')

# ADAPTER_DIR viene de la celda de configuracion de Drive: una sola ruta,
# definida en un solo lugar, para que no se desincronice con el resto.
shutil.copytree(f'{OUTPUT_DIR}/adapter', ADAPTER_DIR, dirs_exist_ok=True)
print(f'Adaptador guardado en {ADAPTER_DIR}')

# Copia fechada: la carpeta sin version se sobrescribe en cada corrida, y por eso
# el adaptador del 27 de septiembre ya no existe. Con esta queda el respaldo.
shutil.copytree(f'{OUTPUT_DIR}/adapter', ADAPTER_DIR_VERSIONADO, dirs_exist_ok=True)
print(f'Respaldo fechado en {ADAPTER_DIR_VERSIONADO}')

In [ ]:
# prepare_model_for_kbit_training() dejo gradient checkpointing activo, lo que fuerza use_cache=False
# durante model.generate() (recalcula toda la atencion en cada token, sin KV-cache). Sin esto, el
# Paso 3 tarda varias veces mas de lo necesario y puede agotar la cuota de GPU de Colab a mitad de la evaluacion.
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

## Paso 3 -- Evaluar el modelo ya afinado (misma validacion)

In [ ]:
finetuned_results = evaluate(model, val_records, label='fine-tuned',
                             checkpoint='finetuned_results.jsonl')
guardar_resultados('finetuned_results.jsonl', finetuned_results)

## Paso 4 -- Comparacion baseline vs. afinado

**Como leerla.** Cada proporcion va con su IC95 de Wilson y la diferencia lleva
`*` cuando la prueba pareada no incluye el 0. Sin eso, cinco ejemplos de 231
parecen un hallazgo.

Dos trampas concretas de esta tabla:

- **"Nombra ruta legal" esta saturada y confundida con la longitud.** El modelo
  base ya venia en 96%, y como la metrica es coincidencia de palabras clave, el
  baseline (que escribe ~5x mas texto) tiene ~5x mas superficie para acertar por
  azar. No sirve para demostrar mejora en ninguna direccion.
- **El desglose por categoria imprime el `n`.** Nueve de las 33 categorias
  tienen `val<=4`, y ahi un "+33%" es UN ejemplo.

Lo que si soporta una conclusion son los dos modos de fabricacion: citas
inventadas y entidades inventadas.


In [ ]:
import pandas as pd

from tools.evaluation.estadistica import diferencia_pareada, proporcion

# Las banderas se recalculan aqui desde el texto generado, en vez de leer las
# que quedaron guardadas en cada registro: si una metrica se corrige despues de
# una corrida, esta tabla refleja la definicion vigente y no una medicion vieja.
filas = []
for etiqueta, datos in [('baseline', baseline_results), ('fine_tuned', finetuned_results)]:
    for r in datos:
        filas.append({'id': r['id'], 'category': r['category'], 'modelo': etiqueta,
                      'nombra_mecanismo': nombra_mecanismo(r['generated']),
                      'cita_no_verificable': cita_no_verificable(r['generated']),
                      'entidad_inventada': bool(entidades_inventadas(r['generated'])),
                      'palabras': len(r['generated'].split())})
df = pd.DataFrame(filas)

N = len(baseline_results)
pares = {}
for col in ('nombra_mecanismo', 'cita_no_verificable', 'entidad_inventada'):
    a = df[df.modelo == 'baseline'].sort_values('id')[col].astype(float).tolist()
    b = df[df.modelo == 'fine_tuned'].sort_values('id')[col].astype(float).tolist()
    pares[col] = diferencia_pareada(a, b)

print('=' * 78)
print(f'RESULTADO PRINCIPAL -- metricas de dominio (N = {N} de validacion)')
print('=' * 78)
print(f"{'metrica':<30}{'baseline':>18}{'afinado':>18}{'diferencia':>12}")
print('-' * 78)
for col, nombre in (('nombra_mecanismo', 'Nombra ruta legal'),
                    ('cita_no_verificable', 'Cita normas inventadas'),
                    ('entidad_inventada', 'Inventa entidades')):
    k_b = int(df[(df.modelo == 'baseline')][col].sum())
    k_f = int(df[(df.modelo == 'fine_tuned')][col].sum())
    iv_b, iv_f, dif = proporcion(k_b, N), proporcion(k_f, N), pares[col]
    marca = '*' if dif.significativo else ' '
    print(f'{nombre:<30}'
          f'{f"{100*k_b/N:.1f}% [{100*iv_b.bajo:.0f}-{100*iv_b.alto:.0f}]":>18}'
          f'{f"{100*k_f/N:.1f}% [{100*iv_f.bajo:.0f}-{100*iv_f.alto:.0f}]":>18}'
          f'{f"{100*dif.valor:+.1f}{marca}":>12}')
print(f"{'Longitud media (palabras)':<30}"
      f"{df[df.modelo=='baseline'].palabras.mean():>18.0f}"
      f"{df[df.modelo=='fine_tuned'].palabras.mean():>18.0f}")
print('-' * 78)
print('[x-y] = IC95 de Wilson.  (*) = la diferencia pareada no incluye el 0.')
print('La similitud contra la referencia se mide en M2 (BLEU/ROUGE/BERTScore),')
print('sobre este mismo split. Aqui no se duplica.')
print('=' * 78)

# --- Desglose por categoria -------------------------------------------------
# Se imprime el n de cada categoria a proposito: 9 de las 33 tienen val<=4, y
# ahi un "+33%" es UN ejemplo. Sin el denominador a la vista, esa columna
# invita a leer ruido como efecto.
por_cat = (df.pivot_table(index='category', columns='modelo', values='nombra_mecanismo', aggfunc='mean')
             .reindex(columns=['baseline', 'fine_tuned']) * 100).round(0)
por_cat['n_val'] = df[df.modelo == 'baseline'].groupby('category').size()
por_cat['interpretable'] = por_cat['n_val'].map(lambda n: 'si' if n >= 8 else 'n muy chico')
print('\nNombra una ruta legal, por categoria:')
print(por_cat.sort_values('n_val').to_string())

df.to_csv(f'{SALIDA_DIR}/metricas_por_registro.csv', index=False)
por_cat.to_csv(f'{SALIDA_DIR}/metricas_por_categoria.csv')
print(f'\nCSV guardados en Drive: evaluacion/')


In [ ]:
# Corte por modo sobre los ejemplos CON contexto.
#
# La tabla de arriba no distingue citar de citar cuando se debia. En la
# corrida del 2026-10-09 el modelo cito en 55 de 55 casos B1 -- el objetivo del
# ciclo -- y tambien en 25 de 35 casos B2, donde el contexto NO responde y lo
# correcto era la frase de escape: escapo en 0. Ese fallo no aparecia en
# ninguna cifra agregada y hubo que encontrarlo leyendo respuestas.
from tools.evaluation.domain_metric import resumen_por_modo

_con_contexto = [r.__dict__ if hasattr(r, '__dict__') else r for r in finetuned_results]
_resumen = resumen_por_modo(_con_contexto)

if not _resumen:
    print('Sin ejemplos con modo: la validacion no trae casos de dataset_v2.')
else:
    print(f"{'modo':6}{'n':>5}{'citan':>8}{'escapan':>10}{'cita no respaldada':>21}")
    print('-' * 50)
    for _m, _d in sorted(_resumen.items()):
        print(f"{_m:6}{_d['n']:>5}{_d['citan']:>8}{_d['escapan']:>10}{_d['no_respaldadas']:>21}")
    print()
    print('B1 deberia citar casi siempre; B2 deberia escapar casi siempre.')
    _b2 = _resumen.get('B2')
    if _b2 and _b2['n'] and _b2['escapan'] / _b2['n'] < 0.5:
        print(f"AVISO: solo {_b2['escapan']} de {_b2['n']} casos B2 se abstuvieron.")
        print('       El modelo responde con contexto que no responde la pregunta.')

## Paso 4b -- Ejemplos reales de respuestas

Las cifras de arriba no se pueden juzgar sin ver que generó el modelo. Esta celda
imprime casos concretos (pregunta, respuesta de referencia, baseline y afinado) y
guarda una tabla en Drive para citarla en la documentación. Se muestran tanto
casos donde el fine-tuning mejoró como casos donde no, para no documentar solo lo
que conviene.

In [ ]:
import pandas as pd

pares = list(zip(baseline_results, finetuned_results))
mejoro = [(b, f) for b, f in pares if f['nombra_mecanismo'] and not b['nombra_mecanismo']]
sin_ruta = [(b, f) for b, f in pares if not f['nombra_mecanismo']]
# Los dos modos de fabricacion, que son el resultado real de M1. Leer estos
# casos es la unica forma de ver QUE se inventa: la cifra sola no lo dice.
cito_normas = [(b, f) for b, f in pares if cita_no_verificable(f['generated'])]
invento_entidad = [(b, f) for b, f in pares if entidades_inventadas(f['generated'])]


def mostrar(titulo, casos, n=3, con_entidades=False):
    print('=' * 78)
    print(titulo)
    print('=' * 78)
    for b, f in casos[:n]:
        print(f"\nCATEGORIA: {b['category']}  (id {b['id']})")
        print(f"PREGUNTA:   {b['messages'][1]['content']}")
        print(f"REFERENCIA: {b['messages'][2]['content']}")
        print(f"BASELINE:   {b['generated']}")
        print(f"AFINADO:    {f['generated']}")
        if con_entidades:
            print(f"  -> marcadas en el afinado: {entidades_inventadas(f['generated'])}")
            print(f"  -> marcadas en el baseline: {entidades_inventadas(b['generated'])}")


mostrar(f'CASOS DONDE EL FINE-TUNING APORTO ({len(mejoro)} en total)', mejoro)
mostrar(f'CASOS DONDE EL AFINADO SIGUE SIN NOMBRAR UNA RUTA ({len(sin_ruta)} en total)', sin_ruta)
mostrar(f'EL AFINADO SE INVENTO UNA ENTIDAD ({len(invento_entidad)} en total)',
        invento_entidad, n=5, con_entidades=True)
mostrar(f'EL AFINADO CITO UNA NORMA POR NUMERO ({len(cito_normas)} en total)', cito_normas, n=5)

tabla = pd.DataFrame([{
    'id': b['id'], 'categoria': b['category'],
    'pregunta': b['messages'][1]['content'],
    'referencia': b['messages'][2]['content'],
    'baseline': b['generated'], 'afinado': f['generated'],
    'baseline_ruta': b['nombra_mecanismo'], 'afinado_ruta': f['nombra_mecanismo'],
    'baseline_entidades': entidades_inventadas(b['generated']),
    'afinado_entidades': entidades_inventadas(f['generated']),
    'baseline_cito': cita_no_verificable(b['generated']),
    'afinado_cito': cita_no_verificable(f['generated']),
} for b, f in pares])
tabla.to_csv(f'{SALIDA_DIR}/ejemplos_baseline_vs_afinado.csv', index=False)
print(f'\n\nTabla completa guardada en Drive: evaluacion/ejemplos_baseline_vs_afinado.csv')


## Paso 4c -- Publicar el adaptador en Hugging Face Hub

El adaptador no puede vivir solo en un Drive privado: sin acceso a ese Drive nadie
puede verificar los resultados ni reproducirlos. Subirlo al Hub lo vuelve
auditable y permite cargarlo desde cualquier parte con su identificador.

Necesitas un token de escritura de https://huggingface.co/settings/tokens.

In [ ]:
from huggingface_hub import login, create_repo, upload_folder

HF_USER = 'Tomas0626'          # ajusta si tu usuario es otro
REPO_ID = f'{HF_USER}/amparo-lora-qwen2.5-7b'

login()   # pega tu token de escritura cuando lo pida

create_repo(REPO_ID, repo_type='model', exist_ok=True)
upload_folder(
    repo_id=REPO_ID,
    folder_path=ADAPTER_DIR,
    commit_message=f'Adaptador LoRA de M1 ({VERSION_ADAPTADOR}, dataset v1+v2: 2291 ejemplos, 33 categorias)',
)
print(f'Adaptador publicado: https://huggingface.co/{REPO_ID}')

## Manifiesto

Con que se entreno: commit, hash del adaptador producido, hash del dataset,
del eval set, y el reparto train/val por origen. Sin esto, dos corridas que
dicen la misma fecha pueden haber entrenado con adaptadores distintos y no
hay como saberlo.


In [ ]:
# Manifiesto de la corrida. M1 era el unico modulo sin uno automatico -- el de
# results/m1_2026-10-06 se escribio a mano -- y es justo el que produce el
# adaptador, el artefacto cuya perdida dejo las corridas del 27 de septiembre
# sin forma de reproducirse.
#
# Registra el hash del adaptador que acaba de salir y el del dataset con el que
# se entreno, no el de dataset_legal.jsonl.
import subprocess

# El hardware solo se registra si esta celda corre EN la sesion que entreno. Si
# se corre despues para completar el manifiesto, nvidia-smi responde igual --
# con la GPU de ESA sesion, que no es la del entrenamiento -- y el manifiesto
# quedaria atribuyendole a la corrida un hardware que no fue. Paso el
# 2026-10-09: se genero en una A100 distinta y el archivo no se marcaba como
# reconstruido. La senal fiable es si las variables del entrenamiento siguen
# vivas.
_entrenando = "trainer" in globals()

_hw = ""
if _entrenando:
    try:
        _hw = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                              "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except FileNotFoundError:
        _hw = ""
else:
    print("Sin 'trainer' en memoria: esta no es la sesion que entreno.")
    print("El manifiesto se marca como reconstruido y no registra hardware.")

_extra = f' --hardware "{_hw}"' if _hw else ""

# La configuracion de la corrida, a un archivo que el manifiesto firma. Sin
# esto el manifiesto dice con que datos se entreno pero no con que ajustes, y
# la corrida no se reproduce. Son los MISMOS valores que v1 a proposito: si
# cambian dos cosas a la vez, una mejora en B2 no se puede atribuir a los pares
# contrastivos.
import json as _json

_hiper = {
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "lora_target_modules": LORA_TARGET_MODULES,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "per_device_batch_size": PER_DEVICE_BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "batch_efectivo": PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS,
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_new_tokens_eval": MAX_NEW_TOKENS_EVAL,
    "random_seed": RANDOM_SEED,
    "iguales_a_v1": True,
}
_ruta_hiper = f"{SALIDA_DIR}/hiperparametros_v2.json"
with open(_ruta_hiper, "w", encoding="utf-8") as f:
    _json.dump(_hiper, f, ensure_ascii=False, indent=2)

# --dataset: v2 entrena con dataset_m1_v3.jsonl. Sin el flag el manifiesto
# firmaria el de v1 y diria que esta corrida uso un archivo que no uso.
!python -m tools.manifiesto_m1 --adaptador "{ADAPTER_DIR}" --salida "{SALIDA_DIR}" --dataset "{DATASET_PATH}" --hiperparametros "{_ruta_hiper}"{_extra}

## Citas: la cifra honesta, separada por contexto

La linea de progreso dice "citas numeradas" y no "inventadas" a proposito:
`has_invented_citation` marca cualquier cita numerada, y 103 de los 334
ejemplos de validacion traen fragmentos de norma en el prompt, donde citar es
lo que se pide. Promediar las dos cosas da un numero que no significa nada.

Esta celda las separa. Es la que hay que leer.


In [ ]:
# Las citas, separadas por si habia contexto. Es la cifra que vale.
#
# `cita_no_verificable` (has_invented_citation) marca cualquier cita numerada.
# Nacio para M1 sin contexto, donde toda cita salia de la memoria del modelo y
# por tanto no se podia verificar. Desde data/dataset_v2.jsonl hay 103 ejemplos
# de validacion que SI traen fragmentos de norma en el prompt, y ahi citar es
# justo lo que se pide.
#
# Promediar las dos cosas sobre los 334 da un numero que no significa nada: el
# 2026-10-09 decia 27.5 % de "citas inventadas" para el afinado, cuando las 92
# que marcaba eran todas de los v2 -- citas correctas, tomadas del contexto --
# y sobre los 231 sin contexto el afinado no cito ni una vez.
#
# Tres cifras distintas, que no se promedian:
#   v1 (231)  cita sin tener de donde: se verifica contra nada -> no verificable
#   v2 (103)  cita teniendo contexto: se verifica contra el contexto -> PROCEDENCIA
#   v2 (103)  si lo citado responde la pregunta -> PERTINENCIA, no lo dicen las
#             reglas; eso lo mide tools/evaluation/criterio_contexto.py en M2.
from tools.evaluation.domain_metric import (
    cita_no_respaldada,
    has_invented_citation,
    resumen_por_modo,
)

# Si la sesion no tiene las generaciones en memoria (p. ej. un runtime CPU
# abierto solo para esto), se leen del repo. Volver a generarlas cuesta horas
# de GPU y no cambia nada: son las mismas respuestas ya guardadas.
def _cargar(nombre, archivo):
    if nombre in globals():
        return globals()[nombre]
    import json as _j
    from pathlib import Path as _P

    ruta = _P("/content/Amparo/results/m1_2026-10-09") / archivo
    return [_j.loads(l) for l in ruta.read_text(encoding="utf-8").splitlines() if l.strip()]


baseline_results = _cargar("baseline_results", "baseline_results.jsonl")
finetuned_results = _cargar("finetuned_results", "finetuned_results.jsonl")

_PARES = [("baseline", baseline_results), ("fine-tuned", finetuned_results)]

print("CITAS, separadas por si la respuesta se genero con contexto")
print()
print(f"{'':12}{'v1 sin contexto':>20}{'v2 con contexto':>20}{'mezclado (no usar)':>21}")
print(f"{'':12}{'(de memoria)':>20}{'(sin respaldo)':>20}{'':>21}")
print("-" * 73)
for _etq, _res in _PARES:
    _v1 = [r for r in _res if not r.get("modo")]
    _v2 = [r for r in _res if r.get("modo")]
    _a = sum(1 for r in _v1 if has_invented_citation(r["generated"]))
    _b = sum(1 for r in _v2 if cita_no_respaldada(r["generated"], r.get("contexto")))
    _m = sum(1 for r in _res if has_invented_citation(r["generated"]))
    print(f"{_etq:12}{f'{_a}/{len(_v1)} = {_a/len(_v1):.1%}':>20}"
          f"{f'{_b}/{len(_v2)} = {_b/len(_v2):.1%}':>20}"
          f"{f'{_m}/{len(_res)} = {_m/len(_res):.1%}':>21}")

print()
print("Y sobre los 103 con contexto, que hizo en cada modo:")
print()
print(f"{'modo':6}{'n':>5}{'citan':>8}{'escapan':>9}{'sin respaldo':>14}")
print("-" * 42)
for _modo, _d in sorted(resumen_por_modo(finetuned_results).items()):
    print(f"{_modo:6}{_d['n']:>5}{_d['citan']:>8}{_d['escapan']:>9}{_d['no_respaldadas']:>14}")

print()
print("Ojo con B2: ahi el contexto NO responde la pregunta, asi que lo correcto")
print("es escapar. Las citas que haga en B2 saldran 'con respaldo' -- el articulo")
print("si estaba en el contexto -- y la respuesta no sirve igual. Esa es la")
print("diferencia entre procedencia y pertinencia, y la mide M2 (fase 4b).")


# --- Integridad de la salida: que la respuesta siga siendo una respuesta -----
#
# Qwen2.5-7B es multilingue con pre-entrenamiento mayormente chino e ingles, y
# el modelo base pierde el idioma a mitad de frase. Aparecio de rebote: tres
# respuestas del baseline no terminaban en punto y parecian truncadas; no lo
# estaban, habian cambiado de alfabeto.
#
# Va aparte de la correccion juridica y de la abstencion, y su denominador son
# los 334 -- no los 231 sin contexto que usan las metricas de citas.
from tools.evaluation.integridad_salida import resumen as _integridad

print()
print("INTEGRIDAD DE SALIDA (poblacion: los 334 de validacion)")
print()
print(f"{'':12}{'n':>5}{'deriva':>9}{'rompe rol':>11}{'turno fict':>12}{'alguno':>9}")
print("-" * 58)
for _etq, _res in _PARES:
    _d = _integridad(_res)
    print(f"{_etq:12}{_d['n']:>5}{_d['deriva_de_idioma']:>9}{_d['rompe_el_rol']:>11}"
          f"{_d['turno_ficticio']:>12}{_d['alguno']:>9}")
    if _d["ids"]:
        print(f"{'':12}ids: {_d['ids']}")

print()
print("Cero observado no es riesgo cero: es la base de regresion. Los cuatro")
print("casos del baseline estan en tests/evaluation/test_integridad_salida.py.")
